# Longitudinal face-aging DataLoaders

Este notebook solo prepara y verifica los datos. Para usar el dataset completo en el servidor, cambia `DATASET_ROOT`; no hay que modificar el paquete ni reconstruir índices dentro de `__getitem__`.

In [ ]:
from pathlib import Path
import sys

# Permite ejecutar el notebook desde notebooks/ o desde la raíz del repositorio.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / '__init__.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data import (
    build_face_aging_dataloaders,
    inspect_batch,
    plot_pair_grid,
    run_data_pipeline_validation,
)

In [ ]:
def load_face_aging_data(dataset_root, *, image_size=256, batch_size=4, num_workers=0, seed=42, cache_dir=None):
    """Construye loaders portables a partir de la carpeta que contiene id_XXXX/."""
    dataset_root = Path(dataset_root).expanduser().resolve()
    cache_dir = Path(cache_dir).expanduser().resolve() if cache_dir else None

    loaders, metadata = build_face_aging_dataloaders(
        root_dir=dataset_root,
        image_size=image_size,
        batch_size=batch_size,
        num_workers=num_workers,          # 0 es ideal para depurar en Jupyter
        split_ratios=(0.80, 0.10, 0.10),
        seed=seed,
        train_pair_strategy='random_target',
        eval_pair_strategy='all',
        min_age_gap=1,
        max_age_gap=None,
        prompt_style='selfage',
        dynamic_person_word=False,
        horizontal_flip_prob=0.0,
        train_drop_last=False,            # útil con el sample pequeño; usar True al entrenar
        pin_memory=False,                 # cambiar a True si se entrena con CUDA
        manifest_path=(cache_dir / 'manifest.csv') if cache_dir else None,
        split_path=(cache_dir / 'splits.csv') if cache_dir else None,
    )
    return loaders, metadata

In [ ]:
# Local: sample incluido. En el servidor, reemplazar por la ruta del dataset completo.
DATASET_ROOT = PROJECT_ROOT / 'data' / 'sample'
# DATASET_ROOT = Path('/ruta/en/el/servidor/dataset_unificado')

loaders, metadata = load_face_aging_data(
    DATASET_ROOT,
    image_size=256,
    batch_size=4,
    num_workers=0,
)
train_loader = loaders['train']
val_loader = loaders['val']
test_loader = loaders['test']

print('Manifest:', metadata['manifest_stats'])
print('Splits:', metadata['split_stats'])
print('Asignación por identidad:', metadata['split_assignments'])

In [ ]:
batch = next(iter(train_loader))
inspect_batch(batch)
# plot_pair_grid(batch, max_pairs=4);

Antes de cada época de entrenamiento, actualiza el epoch para que `random_target` elija otros targets de forma reproducible:

In [ ]:
epoch = 0
train_loader.dataset.set_epoch(epoch)
# for batch in train_loader:
#     ... entrenamiento futuro ...

Auditoría completa opcional (decodifica todas las imágenes y puede tardar con el dataset real):

In [ ]:
validation_report = run_data_pipeline_validation(DATASET_ROOT, seed=42, validate_images=True)
print('PASSED:', validation_report['passed'])
print('ERRORS:', validation_report['errors'])
print('WARNINGS:', validation_report['warnings'])